In [ ]:
"""
Skenario 2: NeuMF + K-Means Cluster (tanpa TPE)
Menggunakan Data: 5 Fitur Audio

Arsitektur IDENTIK dengan Skenario 4:
  - MLP 3 layer: [emb_dim*2 + cluster_dim → 128 → 64 → 32]
  - Cluster embedding disuntikkan HANYA ke MLP input (GMF bersih)
  - Semua hyperparameter FIXED (tidak ada TPE)
  - Evaluasi multi-seed seperti Skenario 1
"""

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path
from tqdm import tqdm
import math
import pickle

# =====================
# CONFIG — FIXED (tidak ada TPE)
# =====================
CLUSTER_PATH = "item_dataset_5f.csv"
ENCODER_PATH = "data/pkl/item_encoder.pkl"

DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS       = 30
BATCH_SIZE   = 1024
LR           = 1e-3
EMB_DIM      = 64
CLUSTER_DIM  = 16
DROPOUT      = 0.2
TOP_K        = 10
NUM_NEG      = 99
RANDOM_STATE = 42
EVAL_SEEDS   = [42, 123, 456]

print("=" * 60)
print("  Skenario 2: NeuMF + K-Means Cluster (tanpa TPE)")
print("  5 Fitur Audio | Fixed Hyperparameter | Multi-Seed Eval")
print("=" * 60)
print("Device:", DEVICE)

# =====================
# LOAD DATA
# =====================
print("\nMemuat dataset...")
train_df = pd.read_csv("data/train_dataset.csv")
test_df  = pd.read_csv("data/test_dataset.csv")
full_df  = pd.read_csv("data/user_dataset_final.csv")

n_users = full_df['user_id_enc'].max() + 1
n_items = full_df['item_id_enc'].max() + 1
print(f"Users: {n_users}, Items: {n_items}")
print(f"Train size: {len(train_df):,}, Test size: {len(test_df):,}")

user_positive_items = (
    train_df[train_df['label'] == 1]
    .groupby('user_id_enc')['item_id_enc']
    .apply(set)
    .to_dict()
)

# =====================
# LOAD CLUSTER
# =====================
print("\nMemuat data cluster (5 Fitur)...")
cluster_df = pd.read_csv(CLUSTER_PATH)
cluster_df['cluster'] = cluster_df['cluster_5f']

with open(ENCODER_PATH, "rb") as f:
    item_encoder = pickle.load(f)

track_to_enc = dict(zip(
    item_encoder.classes_,
    item_encoder.transform(item_encoder.classes_)
))
cluster_df['item_id'] = cluster_df['track_id'].map(track_to_enc)
cluster_df = cluster_df.dropna(subset=['item_id'])
cluster_df['item_id'] = cluster_df['item_id'].astype(int)

unique_clusters       = sorted(cluster_df['cluster'].unique())
cluster_map           = {c: i for i, c in enumerate(unique_clusters)}
cluster_df['cluster'] = cluster_df['cluster'].map(cluster_map)

item_cluster = dict(zip(cluster_df['item_id'], cluster_df['cluster']))
n_clusters   = len(unique_clusters)
print(f"Jumlah cluster  : {n_clusters}")
print(f"Item ter-cluster: {len(item_cluster):,}")

# =====================
# DATALOADER
# =====================
clusters_arr = np.array([item_cluster.get(i, 0) for i in train_df['item_id_enc'].values])

dataset = TensorDataset(
    torch.tensor(train_df['user_id_enc'].values).long(),
    torch.tensor(train_df['item_id_enc'].values).long(),
    torch.tensor(clusters_arr).long(),
    torch.tensor(train_df['label'].values).float()
)
loader = DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=True,
    generator=torch.Generator().manual_seed(RANDOM_STATE)
)
print(f"Training samples: {len(dataset):,}")

# =====================
# MODEL — IDENTIK DENGAN SKENARIO 4
# =====================
class NeuMF(nn.Module):
    def __init__(self, n_users, n_items, n_clusters,
                 emb_dim=64, cluster_dim=16, dropout=0.2):
        super().__init__()
        self.user_gmf    = nn.Embedding(n_users, emb_dim)
        self.item_gmf    = nn.Embedding(n_items, emb_dim)
        self.user_mlp    = nn.Embedding(n_users, emb_dim)
        self.item_mlp    = nn.Embedding(n_items, emb_dim)
        self.cluster_emb = nn.Embedding(n_clusters, cluster_dim)

        mlp_input_dim = emb_dim * 2 + cluster_dim
        self.mlp = nn.Sequential(
            nn.Linear(mlp_input_dim, 128),
            nn.Dropout(dropout),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.Dropout(dropout),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.Dropout(dropout),
            nn.ReLU()
        )
        self.output = nn.Linear(emb_dim + 32, 1)
        self._init_weights()

    def _init_weights(self):
        for emb in [self.user_gmf, self.item_gmf,
                    self.user_mlp, self.item_mlp, self.cluster_emb]:
            nn.init.normal_(emb.weight, std=0.01)
        for layer in self.mlp:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                nn.init.zeros_(layer.bias)
        nn.init.xavier_uniform_(self.output.weight)
        nn.init.zeros_(self.output.bias)

    def forward(self, user, item, cluster):
        gmf    = self.user_gmf(user) * self.item_gmf(item)
        mlp_in = torch.cat([
            self.user_mlp(user),
            self.item_mlp(item),
            self.cluster_emb(cluster)
        ], dim=-1)
        mlp = self.mlp(mlp_in)
        x   = torch.cat([gmf, mlp], dim=-1)
        return self.output(x).squeeze()

torch.manual_seed(RANDOM_STATE)
model     = NeuMF(n_users, n_items, n_clusters,
                  emb_dim=EMB_DIM, cluster_dim=CLUSTER_DIM,
                  dropout=DROPOUT).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
print(f"Jumlah parameter: {sum(p.numel() for p in model.parameters()):,}")

# =====================
# EVALUASI — LOO + 99 NEGATIF (sama persis dengan Skenario 1 & 4)
# =====================
@torch.no_grad()
def evaluate(model, seed=RANDOM_STATE):
    model.eval()
    hits, ndcgs = [], []
    rng = np.random.default_rng(seed)

    test_pos = test_df[test_df['label'] == 1].copy()

    for row in tqdm(test_pos.itertuples(index=False), total=len(test_pos),
                    desc="Evaluating", leave=False):
        user      = int(row.user_id_enc)
        true_item = int(row.item_id_enc)
        positives = user_positive_items.get(user, set())

        negatives = set()
        while len(negatives) < NUM_NEG:
            j = int(rng.integers(n_items))
            if j != true_item and j not in positives:
                negatives.add(j)

        items_eval    = list(negatives) + [true_item]
        users_eval    = [user] * len(items_eval)
        clusters_eval = [item_cluster.get(i, 0) for i in items_eval]

        user_t    = torch.tensor(users_eval).long().to(DEVICE)
        item_t    = torch.tensor(items_eval).long().to(DEVICE)
        cluster_t = torch.tensor(clusters_eval).long().to(DEVICE)

        scores    = torch.sigmoid(model(user_t, item_t, cluster_t)).cpu().numpy()
        rank      = np.argsort(scores)[::-1]
        top_items = np.array(items_eval)[rank[:TOP_K]]

        if true_item in top_items:
            r = int(np.where(top_items == true_item)[0][0]) + 1
            hits.append(1)
            ndcgs.append(1.0 / math.log2(r + 1))
        else:
            hits.append(0)
            ndcgs.append(0.0)

    return np.mean(hits), np.mean(ndcgs)

# =====================
# TRAINING
# =====================
print("\nMemulai Training...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for u, i, c, l in loader:
        u, i, c, l = u.to(DEVICE), i.to(DEVICE), c.to(DEVICE), l.to(DEVICE)
        pred = model(u, i, c)
        loss = criterion(pred, l)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Loss: {total_loss:.4f}")

# =====================
# EVALUASI MULTI-SEED (identik dengan Skenario 1 & 4)
# =====================
print("\nEvaluasi Multi-Seed...")
hr_list, ndcg_list = [], []
for seed in EVAL_SEEDS:
    hr, ndcg = evaluate(model, seed=seed)
    hr_list.append(hr)
    ndcg_list.append(ndcg)
    print(f"  Seed {seed:3d} | HR@{TOP_K}={hr:.4f}, NDCG@{TOP_K}={ndcg:.4f}")

mean_hr   = float(np.mean(hr_list))
mean_ndcg = float(np.mean(ndcg_list))
std_hr    = float(np.std(hr_list))
std_ndcg  = float(np.std(ndcg_list))

print(f"\n{'='*60}")
print(f"  HASIL EVALUASI — Skenario 2: NeuMF + K-Means Cluster")
print(f"{'='*60}")
print(f"  Mean HR@{TOP_K}    : {mean_hr:.4f} ± {std_hr:.4f}")
print(f"  Mean NDCG@{TOP_K}  : {mean_ndcg:.4f} ± {std_ndcg:.4f}")

# =====================
# SIMPAN HASIL
# =====================
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

result_df = pd.DataFrame([{
    "skenario"     : 2,
    "model"        : "NeuMF + K-Means Cluster (5 Fitur Audio)",
    "cluster"      : True,
    "tpe"          : False,
    "HR@10_mean"   : round(mean_hr, 4),
    "HR@10_std"    : round(std_hr, 4),
    "NDCG@10_mean" : round(mean_ndcg, 4),
    "NDCG@10_std"  : round(std_ndcg, 4),
    "seeds"        : str(EVAL_SEEDS),
    "emb_dim"      : EMB_DIM,
    "cluster_dim"  : CLUSTER_DIM,
    "n_clusters"   : n_clusters,
    "dropout"      : DROPOUT,
    "lr"           : LR,
    "batch_size"   : BATCH_SIZE,
    "epochs"       : EPOCHS,
    "notes"        : "MLP 3 layer [128,64,32], Xavier init, cluster 5 fitur, fixed params, multi-seed eval"
}])

result_path = OUTPUT_DIR / "hasil_skenario2_cluster_5f.csv"
result_df.to_csv(result_path, index=False)
print(f"\nHasil disimpan: {result_path}")

Path("models").mkdir(exist_ok=True)
torch.save(model.state_dict(), "models/skenario2_cluster_5f.pt")
print("Model disimpan: models/skenario2_cluster_5f.pt")

  Skenario 2: NeuMF + K-Means Cluster (tanpa TPE)
  5 Fitur Audio | Fixed Hyperparameter | Multi-Seed Eval
Device: cuda

Memuat dataset...
Users: 9529, Items: 4144
Train size: 2,758,490, Test size: 952,900

Memuat data cluster (5 Fitur)...
Jumlah cluster  : 3
Item ter-cluster: 4,144
Training samples: 2,758,490
Jumlah parameter: 1,779,185

Memulai Training...
Epoch 01/30 | Loss: 893.3283
Epoch 02/30 | Loss: 621.0642
Epoch 03/30 | Loss: 493.2834
Epoch 04/30 | Loss: 400.2349
Epoch 05/30 | Loss: 327.4792
Epoch 06/30 | Loss: 272.1914
Epoch 07/30 | Loss: 230.3413
Epoch 08/30 | Loss: 198.1681
Epoch 09/30 | Loss: 171.9851
Epoch 10/30 | Loss: 150.9480
Epoch 11/30 | Loss: 133.2223
Epoch 12/30 | Loss: 118.7541
Epoch 13/30 | Loss: 106.1808
Epoch 14/30 | Loss: 95.8138
Epoch 15/30 | Loss: 86.8297
Epoch 16/30 | Loss: 78.6578
Epoch 17/30 | Loss: 71.6866
Epoch 18/30 | Loss: 65.9518
Epoch 19/30 | Loss: 60.7821
Epoch 20/30 | Loss: 55.7526
Epoch 21/30 | Loss: 51.8617
Epoch 22/30 | Loss: 48.5367
Epoch 23/3

  Seed  42 | HR@10=0.8068, NDCG@10=0.5935


  Seed 123 | HR@10=0.8089, NDCG@10=0.5934


  Seed 456 | HR@10=0.8070, NDCG@10=0.5913

  HASIL EVALUASI — Skenario 2: NeuMF + K-Means Cluster
  Mean HR@10    : 0.8076 ± 0.0009
  Mean NDCG@10  : 0.5928 ± 0.0010

Hasil disimpan: outputs\hasil_skenario2_cluster_5f.csv
Model disimpan: models/skenario2_cluster_5f.pt
